In [ ]:
!pip install inaSpeechSegmenter torchcodec --quiet
!pip install -q -U torch torchaudio pyannote.audio
!pip install -q -U yt-dlp --quiet
!apt-get install -y -qq nodejs ffmpeg

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
WORKING_DIR = '/content/drive/My Drive/Siboubou/'
AUDIO_WORKING_DIR = os.path.join(WORKING_DIR, "siboubou_audios")
AUDIO_FILE_PATH = os.path.join(AUDIO_WORKING_DIR, "original.wav")
SILENCE_MAPPER = os.path.join(AUDIO_WORKING_DIR, "silence_mapper")
SPEAKERS_MAPPER = os.path.join(AUDIO_WORKING_DIR, "speakers_mapper")
YOUTUBE_URL = "https://www.youtube.com/watch?v=BETMN6Jj4Lk"

os.makedirs(AUDIO_WORKING_DIR, exist_ok=True)

In [ ]:
import os
import shutil
from google.colab import files

if not os.path.exists(AUDIO_FILE_PATH):
  uploaded = files.upload()
  if uploaded:
    uploaded_name = next(iter(uploaded))
    if uploaded_name != "cookies.txt":
      shutil.move(uploaded_name, "cookies.txt")
  !yt-dlp --cookies cookies.txt -x --audio-format wav --audio-quality 0 -o "{AUDIO_FILE_PATH}" "{YOUTUBE_URL}"

In [ ]:
#Mapping Silence chunks
import tensorflow as tf
from inaSpeechSegmenter import Segmenter

if not os.path.exists(SILENCE_MAPPER):
  gpus = tf.config.list_physical_devices('GPU')

  seg = Segmenter()
  segmentation = seg(AUDIO_FILE_PATH)

  file = open(SILENCE_MAPPER, 'w')

  for label, start, stop in segmentation:
      if label == "noEnergy":
          file.write(f"{start:.2f}->{stop:.2f}\n")

  file.close()

In [ ]:
#Remove Silence
import os
import subprocess

SILENCE_DIR = os.path.join(AUDIO_WORKING_DIR, "silence")
NO_SILENCE_AUDIO = os.path.join(SILENCE_DIR, "no_silence.wav")
os.makedirs(SILENCE_DIR, exist_ok=True)

with open(SILENCE_MAPPER) as f:
    silence = [tuple(map(float, line.strip().split("->"))) for line in f if line.strip()]

silence_expr = "+".join(f"between(t,{s},{e})" for s, e in silence)
keep_expr = f"not({silence_expr})" if silence_expr else "1"

if not os.path.exists(NO_SILENCE_AUDIO):
  subprocess.run([
      "ffmpeg", "-y", "-i", AUDIO_FILE_PATH,
      "-af", f"aselect='{keep_expr}',asetpts=N/SR/TB",
      NO_SILENCE_AUDIO
  ], check=True)

In [ ]:
#Speakers recognition
import torch
from pyannote.audio import Pipeline
from pyannote.audio.pipelines.utils.hook import ProgressHook
from google.colab import userdata

if not os.path.exists(SPEAKERS_MAPPER):

  pipeline = Pipeline.from_pretrained(
      "pyannote/speaker-diarization-community-1",
      token=userdata.get('HUGGINGFACE_ACCESS_TOKEN')
  )

  if torch.cuda.is_available():
      pipeline.to(torch.device("cuda"))

  with ProgressHook() as hook:
      output = pipeline(NO_SILENCE_AUDIO, hook=hook)

  file = open(SPEAKERS_MAPPER, 'w')
  for turn, speaker in output.speaker_diarization:
      file.write(f"speaker={speaker}, start={turn.start:.2f}, stop={turn.end:.2f}\n")
  file.close()

In [ ]:
#Slice audio
import os
import re
import subprocess
from collections import defaultdict

SPEAKERS_DIR = os.path.join(AUDIO_WORKING_DIR, "speakers")
os.makedirs(SPEAKERS_DIR, exist_ok=True)

turns_by_speaker = defaultdict(list)
order = []
pattern = re.compile(r"speaker=([^,]+), start=([\d.]+), stop=([\d.]+)")

with open(SPEAKERS_MAPPER) as f:
    for line in f:
        m = pattern.search(line)
        if not m:
            continue
        label, start, stop = m.group(1), float(m.group(2)), float(m.group(3))
        if label not in turns_by_speaker:
            order.append(label)
        turns_by_speaker[label].append((start, stop))

for i, label in enumerate(order, start=1):
    expr = "+".join(f"between(t,{s},{e})" for s, e in turns_by_speaker[label])
    out_path = os.path.join(SPEAKERS_DIR, f"speaker_{i}.wav")
    subprocess.run([
        "ffmpeg", "-y", "-i", NO_SILENCE_AUDIO,
        "-af", f"aselect='{expr}',asetpts=N/SR/TB",
        out_path
    ], check=True)
    print(f"speaker_{i}.wav <- {label}")